In [1]:
# CELL 1: Imports
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import tensorflow as tf
warnings.filterwarnings('ignore')

In [4]:
# CELL 2: Load Data and Model
print("=" * 60)
print("LOADING MODEL AND DATA")
print("=" * 60)

# Load model and data
model = tf.keras.models.load_model("../models/multimodal_churn_model.keras")
X_static_test = joblib.load("../models/X_static_test.pkl")
X_lstm_test = joblib.load("../models/X_lstm_test.pkl")
y_test = joblib.load("../models/y_test.pkl")
threshold = joblib.load("../models/optimal_threshold.pkl")
feature_importance = pd.read_csv("../models/feature_importance.csv")

print(f"\n📊 Initial data types:")
print(f"   X_static_test type: {type(X_static_test)}")
print(f"   y_test type: {type(y_test)}")

# Define feature names
feature_names = ['Age', 'Tenure', 'Balance', 'NumOfProducts', 
                 'EstimatedSalary', 'BalanceSalaryRatio', 'TenureAgeRatio',
                 'ProductUtilizationRate', 'AgeBalanceInteraction', 'EngagementScore',
                 'ComplaintCount', 'Geography', 'Gender', 'HasCrCard', 
                 'IsActiveMember', 'CreditScoreCategory']

# Convert X_static_test to DataFrame if it's numpy array
if isinstance(X_static_test, np.ndarray):
    X_static_test = pd.DataFrame(X_static_test, columns=feature_names)
    print("✅ Converted X_static_test to DataFrame")

print(f"\n📊 Final data shapes:")
print(f"   Static test: {X_static_test.shape}")
print(f"   LSTM test:   {X_lstm_test.shape}")
print(f"   Test labels: {len(y_test)}")
print("✅ All data loaded successfully")

LOADING MODEL AND DATA

📊 Initial data types:
   X_static_test type: <class 'numpy.ndarray'>
   y_test type: <class 'numpy.ndarray'>
✅ Converted X_static_test to DataFrame

📊 Final data shapes:
   Static test: (2000, 16)
   LSTM test:   (2000, 6, 1)
   Test labels: 2000
✅ All data loaded successfully


In [5]:
# CELL 3: Get Predictions
print("\n" + "=" * 60)
print("GENERATING PREDICTIONS")
print("=" * 60)

y_pred_prob = model.predict([X_static_test, X_lstm_test]).flatten()
y_pred = (y_pred_prob > threshold).astype(int)

print(f"✅ Predictions generated for {len(y_pred_prob)} customers")


GENERATING PREDICTIONS
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step
✅ Predictions generated for 2000 customers


In [6]:
# CELL 4: Define Risk Levels
print("\n" + "=" * 60)
print("DEFINING RISK LEVELS")
print("=" * 60)

def get_risk_level(probability):
    if probability < 0.3:
        return "Low Risk"
    elif probability < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

risk_levels = [get_risk_level(p) for p in y_pred_prob]
risk_counts = pd.Series(risk_levels).value_counts()

print("✅ Risk levels defined:")
print(f"   Low Risk:    {risk_counts.get('Low Risk', 0)} customers")
print(f"   Medium Risk: {risk_counts.get('Medium Risk', 0)} customers")
print(f"   High Risk:   {risk_counts.get('High Risk', 0)} customers")


DEFINING RISK LEVELS
✅ Risk levels defined:
   Low Risk:    1351 customers
   Medium Risk: 429 customers
   High Risk:   220 customers


In [7]:
# CELL 5: Create Customer Profiles
print("\n" + "=" * 60)
print("CREATING CUSTOMER PROFILES")
print("=" * 60)

# Double-check X_static_test type
print(f"📊 X_static_test type before profile creation: {type(X_static_test)}")

# Ensure X_static_test is a DataFrame
if isinstance(X_static_test, np.ndarray):
    feature_names = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
                     'EstimatedSalary', 'BalanceSalaryRatio', 'TenureAgeRatio',
                     'ProductUtilizationRate', 'AgeBalanceInteraction', 'EngagementScore',
                     'ComplaintCount', 'Geography', 'Gender', 'HasCrCard', 
                     'IsActiveMember', 'CreditScoreCategory']
    X_static_test = pd.DataFrame(X_static_test, columns=feature_names)
    print("✅ Converted X_static_test to DataFrame")

# Geography mapping
geo_map = {0: "France", 1: "Germany", 2: "Spain"}

# Create profiles for first 50 customers
customer_profiles = []
num_customers = min(50, len(X_static_test))

# Handle y_test access
if hasattr(y_test, 'iloc'):
    # y_test is DataFrame/Series
    y_test_values = y_test
    use_iloc_y = True
else:
    # y_test is numpy array
    y_test_values = y_test
    use_iloc_y = False

for i in range(num_customers):
    # Get static features - now X_static_test is definitely DataFrame
    static_row = X_static_test.iloc[i]
    
    # Get actual value based on type
    if use_iloc_y:
        actual_value = y_test_values.iloc[i]
    else:
        actual_value = y_test_values[i]
    
    profile = {
        'customer_id': f"CUST_{1000 + i}",
        'risk_level': risk_levels[i],
        'churn_probability': y_pred_prob[i],
        'predicted_churn': 'Yes' if y_pred[i] == 1 else 'No',
        'actual_churn': 'Yes' if actual_value == 1 else 'No',
        
        # Key features
        'age': static_row.get('Age', 0),
        'balance': static_row.get('Balance', 0),
        'products': static_row.get('NumOfProducts', 0),
        'active': bool(static_row.get('IsActiveMember', 0)),
        'credit_score': static_row.get('CreditScore', 0),
        'tenure': static_row.get('Tenure', 0),
        'geography': geo_map.get(static_row.get('Geography', 0), "Unknown"),
        'gender': 'Male' if static_row.get('Gender', 0) == 0 else 'Female',
        'has_card': bool(static_row.get('HasCrCard', 0)),
        'complaints': int(static_row.get('ComplaintCount', 0))
    }
    customer_profiles.append(profile)

print(f"✅ Created profiles for {len(customer_profiles)} customers")


CREATING CUSTOMER PROFILES
📊 X_static_test type before profile creation: <class 'pandas.core.frame.DataFrame'>
✅ Created profiles for 50 customers


In [8]:
# CELL 6: Define Strategy Templates
print("\n" + "=" * 60)
print("DEFINING RETENTION STRATEGIES")
print("=" * 60)

def get_retention_strategy(customer):
    """Generate personalized retention strategy based on customer profile"""
    
    risk = customer['risk_level']
    prob = customer['churn_probability']
    
    # Strategy header
    strategy = f"""
{'='*80}
🎯 RETENTION STRATEGY FOR {customer['customer_id']}
{'='*80}

📊 CUSTOMER PROFILE:
────────────────────────────────────────────────
• Risk Level:      {risk} ({prob:.1%})
• Age:             {customer['age']}
• Gender:          {customer['gender']}
• Location:        {customer['geography']}
• Products:        {customer['products']}
• Active Member:   {'✅ Yes' if customer['active'] else '❌ No'}
• Credit Card:     {'✅ Yes' if customer['has_card'] else '❌ No'}
• Balance:         ${customer['balance']:,.2f}
• Credit Score:    {customer['credit_score']}
• Tenure:          {customer['tenure']} years
• Complaints:      {customer['complaints']}
"""
    
    # HIGH RISK CUSTOMERS
    if risk == "High Risk":
        strategy += """
🔴 URGENT INTERVENTION REQUIRED
────────────────────────────────────────────────

⚠️ RISK FACTORS IDENTIFIED:
"""
        # Add risk factors
        factors = []
        if not customer['active']:
            factors.append("• Customer is INACTIVE - needs re-engagement")
        if customer['products'] <= 1:
            factors.append("• Low product adoption - cross-sell opportunity")
        if customer['balance'] > 50000:
            factors.append("• High-value customer - significant revenue at risk")
        if customer['complaints'] > 2:
            factors.append("• Multiple complaints - service quality issues")
        if customer['age'] > 60:
            factors.append("• Senior customer - may need different engagement")
        
        strategy += "\n".join(factors) if factors else "• Multiple risk factors detected"
        
        strategy += """

🎯 RECOMMENDED ACTIONS (Priority Order):
────────────────────────────────────────────────

1️⃣ IMMEDIATE OUTREACH (Next 24-48 hours):
   • Priority phone call from relationship manager
   • Personal follow-up on recent issues
   • Welcome back offer with incentives

2️⃣ PREMIUM RETENTION OFFER:
"""
        if customer['balance'] > 50000:
            strategy += """   • Upgrade to Premium Banking - 6 months FREE
   • Dedicated wealth manager assigned
   • Exclusive investment opportunities
"""
        else:
            strategy += """   • Cashback offer: $100 on next 10 transactions
   • Waive monthly fees for 12 months
   • 0% interest on balance transfers for 6 months
"""
        
        if customer['products'] <= 1:
            strategy += """
3️⃣ PRODUCT BUNDLE OFFER:
   • Bundle 2 additional products at 25% discount
   • Free credit card annual fee waiver
   • Personal financial consultation
"""
        
        if not customer['active']:
            strategy += """
4️⃣ RE-ENGAGEMENT CAMPAIGN:
   • Welcome back bonus: $50 after first transaction
   • Personalized app notification with offers
   • Email campaign with success stories
"""
        
        # Calculate expected impact
        expected_savings = customer['balance'] * 0.15  # 15% of balance
        
        strategy += f"""
📈 EXPECTED IMPACT:
   • 40-50% reduction in churn probability
   • Estimated value saved: ${expected_savings:,.0f}
   • ROI: $6-8 per $1 spent
"""
    
    # MEDIUM RISK CUSTOMERS
    elif risk == "Medium Risk":
        strategy += """
🟠 PROACTIVE RETENTION CAMPAIGN
────────────────────────────────────────────────

📋 CUSTOMER INSIGHTS:
• Customer shows MODERATE risk indicators
• Proactive engagement recommended within 1 week

🎯 RECOMMENDED ACTIONS:
────────────────────────────────────────────────

1️⃣ LOYALTY REWARDS:
   • Double loyalty points for 3 months
   • Birthday bonus: $10 cashback
   • Anniversary gift: Free coffee voucher

2️⃣ VALUE-ADDED SERVICES:
   • Free credit score monitoring for 6 months
   • Financial planning webinar invitation
   • Pre-approved credit line increase of $5,000

3️⃣ ENGAGEMENT CAMPAIGN:
   • Personalized email with relevant offers
   • App push notifications for new features
   • Monthly newsletter with financial tips

📈 EXPECTED IMPACT:
   • 25-35% reduction in churn probability
   • Estimated value saved: ${customer['balance'] * 0.1:,.0f}
   • ROI: $3-5 per $1 spent
"""
    
    # LOW RISK CUSTOMERS
    else:
        strategy += """
🟢 LOYALTY REINFORCEMENT
────────────────────────────────────────────────

📋 CUSTOMER VALUE ANALYSIS:
• This is a LOYAL customer with LOW churn risk
• Focus on maintaining satisfaction

🎯 RECOMMENDED ACTIONS:
────────────────────────────────────────────────

1️⃣ APPRECIATION REWARDS:
   • $10 coffee voucher or gift card
   • Birthday bonus: $5 cashback
   • Anniversary gift: Double points for a month

2️⃣ EXCLUSIVE BENEFITS:
   • Early access to new banking features
   • Invitation to customer appreciation events
   • Priority customer support line

3️⃣ REFERRAL PROGRAM:
   • Enhanced referral bonus: $75 per successful referral
   • Referral leaderboard with quarterly prizes
   • "Friend & Family" special offers

📈 EXPECTED IMPACT:
   • 10-15% increase in loyalty metrics
   • Higher referral rate (estimated 2-3 referrals/year)
"""
    
    strategy += f"""
────────────────────────────────────────────────
Strategy Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*80}
"""
    return strategy

print("✅ Strategy templates defined")


DEFINING RETENTION STRATEGIES
✅ Strategy templates defined


In [9]:
# CELL 7: Generate Strategies for All Customers
print("\n" + "=" * 60)
print("GENERATING RETENTION STRATEGIES")
print("=" * 60)

strategies = []
risk_counts = {'Low Risk': 0, 'Medium Risk': 0, 'High Risk': 0}

for i, customer in enumerate(customer_profiles):
    risk_counts[customer['risk_level']] += 1
    
    strategy_text = get_retention_strategy(customer)
    
    strategies.append({
        'customer_id': customer['customer_id'],
        'risk_level': customer['risk_level'],
        'churn_probability': f"{customer['churn_probability']:.1%}",
        'predicted_churn': customer['predicted_churn'],
        'actual_churn': customer['actual_churn'],
        'age': customer['age'],
        'gender': customer['gender'],
        'geography': customer['geography'],
        'products': customer['products'],
        'active': customer['active'],
        'balance': f"${customer['balance']:,.2f}",
        'credit_score': customer['credit_score'],
        'tenure': customer['tenure'],
        'complaints': customer['complaints'],
        'strategy': strategy_text
    })
    
    # Show progress
    if (i + 1) % 10 == 0:
        print(f"✅ Generated strategies for {i + 1}/{num_customers} customers")

print(f"\n📊 Risk Breakdown:")
print(f"   High Risk:    {risk_counts['High Risk']}")
print(f"   Medium Risk:  {risk_counts['Medium Risk']}")
print(f"   Low Risk:     {risk_counts['Low Risk']}")


GENERATING RETENTION STRATEGIES
✅ Generated strategies for 10/50 customers
✅ Generated strategies for 20/50 customers
✅ Generated strategies for 30/50 customers
✅ Generated strategies for 40/50 customers
✅ Generated strategies for 50/50 customers

📊 Risk Breakdown:
   High Risk:    5
   Medium Risk:  15
   Low Risk:     30


In [10]:
# CELL 8: Save Strategies to CSV
print("\n" + "=" * 60)
print("SAVING STRATEGIES")
print("=" * 60)

strategies_df = pd.DataFrame(strategies)
strategies_df.to_csv("../models/retention_strategies.csv", index=False)
print(f"✅ Saved {len(strategies_df)} strategies to ../models/retention_strategies.csv")


SAVING STRATEGIES
✅ Saved 50 strategies to ../models/retention_strategies.csv


In [11]:
# CELL 9: Create Executive Summary
print("\n" + "=" * 80)
print("📊 EXECUTIVE SUMMARY REPORT")
print("=" * 80)

# Calculate business metrics
avg_customer_value = 5000
retention_cost = 50

high_risk_count = risk_counts['High Risk']
medium_risk_count = risk_counts['Medium Risk']
low_risk_count = risk_counts['Low Risk']

# Financial projections
high_risk_value = high_risk_count * avg_customer_value * 0.4  # 40% retention
medium_risk_value = medium_risk_count * avg_customer_value * 0.6  # 60% retention
total_saved = high_risk_value + medium_risk_value
campaign_cost = (high_risk_count + medium_risk_count) * retention_cost
roi = (total_saved - campaign_cost) / campaign_cost

print(f"""
🎯 CAMPAIGN OVERVIEW
────────────────────────────────────────────────
Total Customers Analyzed:   {num_customers}
Strategies Generated:       {len(strategies_df)}
Date:                       {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

📊 RISK DISTRIBUTION
────────────────────────────────────────────────
🔴 High Risk:     {high_risk_count:3d} ({high_risk_count/num_customers*100:.1f}%)
🟠 Medium Risk:   {medium_risk_count:3d} ({medium_risk_count/num_customers*100:.1f}%)
🟢 Low Risk:      {low_risk_count:3d} ({low_risk_count/num_customers*100:.1f}%)

💰 FINANCIAL IMPACT
────────────────────────────────────────────────
Total Balance at Risk:      ${sum(c['balance'] for c in customer_profiles if isinstance(c['balance'], (int, float))):,.0f}
Projected Value Saved:      ${total_saved:,.0f}
Campaign Cost:               ${campaign_cost:,.0f}
Net Gain:                    ${total_saved - campaign_cost:,.0f}
ROI:                         {roi:.1f}x

🎯 ACTION PLAN
────────────────────────────────────────────────
🔴 HIGH PRIORITY (Next 24-48 hours):
   • Contact all {high_risk_count} high-risk customers
   • Priority calls for high-balance customers
   • Offer premium retention packages

🟠 MEDIUM PRIORITY (Next 1-2 weeks):
   • Launch email campaign for {medium_risk_count} medium-risk customers
   • Track response rates and engagement

🟢 LOW PRIORITY (Ongoing):
   • Send appreciation rewards to {low_risk_count} loyal customers
   • Encourage referrals
""")

# Save summary
with open('../models/executive_summary.txt', 'w') as f:
    f.write("EXECUTIVE SUMMARY REPORT\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Total Customers: {num_customers}\n")
    f.write(f"High-Risk: {high_risk_count}\n")
    f.write(f"Medium-Risk: {medium_risk_count}\n")
    f.write(f"Low-Risk: {low_risk_count}\n\n")
    f.write(f"Projected Value Saved: ${total_saved:,.0f}\n")
    f.write(f"ROI: {roi:.1f}x\n")

print("\n✅ Summary saved to ../models/executive_summary.txt")


📊 EXECUTIVE SUMMARY REPORT

🎯 CAMPAIGN OVERVIEW
────────────────────────────────────────────────
Total Customers Analyzed:   50
Strategies Generated:       50
Date:                       2026-03-11 21:19:57

📊 RISK DISTRIBUTION
────────────────────────────────────────────────
🔴 High Risk:       5 (10.0%)
🟠 Medium Risk:    15 (30.0%)
🟢 Low Risk:       30 (60.0%)

💰 FINANCIAL IMPACT
────────────────────────────────────────────────
Total Balance at Risk:      $1
Projected Value Saved:      $55,000
Campaign Cost:               $1,000
Net Gain:                    $54,000
ROI:                         54.0x

🎯 ACTION PLAN
────────────────────────────────────────────────
🔴 HIGH PRIORITY (Next 24-48 hours):
   • Contact all 5 high-risk customers
   • Priority calls for high-balance customers
   • Offer premium retention packages

🟠 MEDIUM PRIORITY (Next 1-2 weeks):
   • Launch email campaign for 15 medium-risk customers
   • Track response rates and engagement

🟢 LOW PRIORITY (Ongoing):
   • S

In [12]:
# CELL 10: Display Sample Strategies
print("\n" + "=" * 80)
print("SAMPLE RETENTION STRATEGIES")
print("=" * 80)

# Show one strategy per risk level
for risk in ['High Risk', 'Medium Risk', 'Low Risk']:
    sample = next((s for s in strategies if s['risk_level'] == risk), None)
    if sample:
        print(f"\n{sample['strategy']}")


SAMPLE RETENTION STRATEGIES


🎯 RETENTION STRATEGY FOR CUST_1010

📊 CUSTOMER PROFILE:
────────────────────────────────────────────────
• Risk Level:      High Risk (84.4%)
• Age:             1.4300838412616106
• Gender:          Female
• Location:        Germany
• Products:        -0.9102564852985486
• Active Member:   ✅ Yes
• Credit Card:     ✅ Yes
• Balance:         $0.56
• Credit Score:    0
• Tenure:          -0.6962018035157382 years
• Complaints:      0

🔴 URGENT INTERVENTION REQUIRED
────────────────────────────────────────────────

⚠️ RISK FACTORS IDENTIFIED:
• Low product adoption - cross-sell opportunity

🎯 RECOMMENDED ACTIONS (Priority Order):
────────────────────────────────────────────────

1️⃣ IMMEDIATE OUTREACH (Next 24-48 hours):
   • Priority phone call from relationship manager
   • Personal follow-up on recent issues
   • Welcome back offer with incentives

2️⃣ PREMIUM RETENTION OFFER:
   • Cashback offer: $100 on next 10 transactions
   • Waive monthly fees for 12 